# Ethiopia Food Prices  08 FEATURE ENGINEERING

Create a single monthly feature table for downstream forecasting and spike prediction.


In [ ]:
# This notebook creates a broad, reusable feature table for the project.
# It combines many markets and commodities into one monthly dataset so later notebooks
# can train models on a larger population of price patterns.
# This is different from the single-series story we use in the time-series and ML notebooks.
# Those notebooks focus on one representative case: Addis Ababa -> Maize (white) -> Wholesale -> 100 KG.
# So the feature table is global and project-wide, while the forecasting case is a specific example.
# Both are valid, but they answer different questions.
#
# Load the spike-flagged dataset produced in the previous notebook.
# We keep only retail food rows because this modeling stage focuses on food-price dynamics.
import pandas as pd
from pathlib import Path

root = Path('../')
spikes = pd.read_csv(root / 'data' / 'eth_food_prices_spikes.csv', parse_dates=['date'])
spikes = spikes[(spikes['category'] != 'non-food') & (spikes['pricetype'] == 'Retail')].copy()
spikes['date'] = pd.to_datetime(spikes['date'])
spikes = spikes.sort_values(['market', 'commodity', 'date']).reset_index(drop=True)

print(f'Loaded {len(spikes):,} rows')


Loaded 51,071 rows


In [13]:
# Create time and forecasting features.
# Lag features capture recent price history; rolling means/stds capture trend and volatility.
# These signals help distinguish normal seasonal movement from abnormal price spikes.
spikes['month'] = spikes['date'].dt.to_period('M').dt.to_timestamp()
spikes['year'] = spikes['date'].dt.year
spikes['month_of_year'] = spikes['date'].dt.month

# Lag features: how much the price was 1, 3, and 6 months ago.
spikes['lag_1'] = spikes.groupby(['market', 'commodity'])['usdprice'].transform(lambda s: s.shift(1))
spikes['lag_3'] = spikes.groupby(['market', 'commodity'])['usdprice'].transform(lambda s: s.shift(3))
spikes['lag_6'] = spikes.groupby(['market', 'commodity'])['usdprice'].transform(lambda s: s.shift(6))

# Rolling statistics smooth short-term noise and highlight abnormal movements.
for window in [3, 6, 12]:
    rolling_mean = spikes.groupby(['market', 'commodity'])['usdprice'].transform(
        lambda s: s.shift(1).rolling(window=window, min_periods=1).mean()
    )
    rolling_std = spikes.groupby(['market', 'commodity'])['usdprice'].transform(
        lambda s: s.shift(1).rolling(window=window, min_periods=2).std()
    )
    spikes[f'rolling_mean_{window}'] = rolling_mean
    spikes[f'rolling_std_{window}'] = rolling_std


# Month-over-month percent change and relative price level help the model detect shocks.
spikes['pct_change_1m'] = spikes.groupby(['market', 'commodity'])['usdprice'].transform(
    lambda s: s.pct_change().shift(1) * 100
)
spikes['price_to_mean_ratio'] = (
    spikes['usdprice'] / spikes[['market', 'commodity', 'usdprice']]
    .groupby(['market', 'commodity'])['usdprice']
    .transform('mean')
)
spikes['is_spike_flag'] = spikes['is_spike'].astype(int)
print('Feature columns prepared successfully.')

Feature columns prepared successfully.


In [14]:
# Final feature table for modeling.
# We keep only the most informative columns and remove rows without enough history,
# so the downstream models train on complete and comparable inputs.
feature_df = spikes[[
    'date', 'market', 'commodity', 'commodity_unit', 'admin1', 'year', 'month_of_year',
    'usdprice', 'lag_1', 'lag_3', 'lag_6', 'rolling_mean_3', 'rolling_mean_6', 'rolling_mean_12',
    'rolling_std_3', 'rolling_std_6', 'rolling_std_12', 'pct_change_1m', 'price_to_mean_ratio',
    'price_outlier_flag', 'is_spike_flag', 'is_drop'
]].dropna().copy()

# Convert text columns explicitly to strings for stable downstream training.
feature_df['market'] = feature_df['market'].astype(str)
feature_df['commodity'] = feature_df['commodity'].astype(str)
feature_df['admin1'] = feature_df['admin1'].astype(str)
feature_df['commodity_unit'] = feature_df['commodity_unit'].astype(str)

# Save the engineered dataset so the forecasting and classification notebooks can reuse it.
feature_path = root / 'data' / 'eth_food_prices_features.csv'
feature_df.to_csv(feature_path, index=False)
print(f'Saved features to {feature_path}')
print(feature_df.head(5).to_string(index=False))


Saved features to ..\data\eth_food_prices_features.csv
      date market          commodity          commodity_unit admin1  year  month_of_year  usdprice  lag_1  lag_3  lag_6  rolling_mean_3  rolling_mean_6  rolling_mean_12  rolling_std_3  rolling_std_6  rolling_std_12  pct_change_1m  price_to_mean_ratio  price_outlier_flag  is_spike_flag  is_drop
2026-07-15 Abaala Butter (goat milk) Butter (goat milk) (KG)   Afar  2026              7      9.35   9.38   9.53   5.77        9.470000        8.905000         8.905000       0.079373       1.538555        1.538555      -1.263158             1.042529               False              0    False
2026-06-15 Abaala             Garlic             Garlic (KG)   Afar  2026              6      2.38   2.53   2.56   4.37        2.543333        2.855000         2.855000       0.015275       0.742341        0.742341      -0.393701             0.865455               False              0    False
2026-07-15 Abaala             Garlic             Garlic (KG)